[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/39_ppo_loss_solution.ipynb)

# 参考解答：近端策略优化裁剪损失（PPO）

PPO 裁剪代理损失的参考解答。


In [ ]:
# 在 Colab 中安装 torch-judge（JupyterLab/Docker 中会跳过）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn.functional as F
from torch import Tensor


In [ ]:
# ✅ 参考解答

def ppo_loss(new_logps: Tensor, old_logps: Tensor, advantages: Tensor,
             clip_ratio: float = 0.2) -> Tensor:
    """PPO 裁剪代理损失。

    new_logps: (B,) 当前策略的对数概率
    old_logps: (B,) 旧策略的对数概率（视为常量）
    advantages: (B,) 优势估计（视为常量）
    返回：标量损失（Tensor）
    """
    # 分离 old_logps 和 advantages，使梯度仅通过 new_logps 传播
    old_logps_detached = old_logps.detach()
    adv_detached = advantages.detach()

    # 在对数空间计算重要性采样比率 r = pi_new / pi_old
    ratios = torch.exp(new_logps - old_logps_detached)

    # 未裁剪和裁剪后的目标
    unclipped = ratios * adv_detached
    clipped = torch.clamp(ratios, 1.0 - clip_ratio, 1.0 + clip_ratio) * adv_detached

    # PPO 目标：对更保守的目标取负均值
    return -torch.min(unclipped, clipped).mean()


In [ ]:
# 示例
new_logps = torch.tensor([0.0, -0.2, -0.4, -0.6])
old_logps = torch.tensor([0.0, -0.1, -0.5, -0.5])
advantages = torch.tensor([1.0, -1.0, 0.5, -0.5])
print('Loss:', ppo_loss(new_logps, old_logps, advantages, clip_ratio=0.2))


In [ ]:
from torch_judge import check
check('ppo_loss')
